# Spanish tutor: the evaluation framework

How well does the tutor do what it claims? This notebook re-creates every result of
roadmap slice 4.4 from what was stored: the learning log, the benchmark runs, the judges'
ratings and Jason's own. **It makes no model calls**: every judgment was made once, by the
commands in `python -m spanish_tutor.evaluation`, and kept (append-only) in the
`eval_runs`, `eval_items` and `ratings` tables.

| Section | Question | Source |
|---|---|---|
| 1. The log | Does the tutor stay inside the vocabulary, and teach what it uses? | Jason's real sessions, in SQL |
| 2. The benchmark | Does the grader catch learner mistakes, and tell form from word? | 40 scripted messages, 24 planted mistakes |
| 3. The new-word detector | When the tutor teaches a word as new, is it new? | Jason's ratings of 13 taught words |
| 4. Translation judges | Can a model judge translations? Consistently? Like the learner would? | 88 lines × 5 repeats, 18 attempts × 5, 28 of Jason's ratings |
| 5. Retrieval | Is what the tutor retrieves relevant, and are its replies faithful to it? | RAGAS's two metrics, implemented directly |

**A principle throughout:** samples are small (two real conversations so far), so every rate
comes with its *n* and a 95% Wilson interval, and nothing judged by a model is reported
before the judge has been checked: first against itself (repeats), then against a person.

**Before running:** the data pipeline (notebook 01) must have been built, and the evaluation
commands run. The outputs include Jason's personal ratings; the nbstripout git filter
removes outputs from commits.

In [ ]:
import json
import sqlite3
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import pandas as pd

from spanish_tutor.config import DB_PATH
from spanish_tutor.evaluation import benchmark, metrics, ratings, retrieval, translation
from spanish_tutor.progress import query

pd.set_option("display.max_colwidth", 80)

# Read-only: the evaluation never writes to the word bank it measures.
conn = sqlite3.connect(f"file:{DB_PATH.as_posix()}?mode=ro", uri=True)
conn.row_factory = sqlite3.Row


pd.read_sql_query(
    "SELECT run_id, kind, model, prompt_version, started_at FROM eval_runs ORDER BY run_id", conn
).set_index("run_id")

## 1. The log: adherence and teaching completeness

Every metric is a hand-written SQL query in `sql/queries/eval_*.sql`, over the same tables
the app writes, so the numbers can't drift from the log. The same queries also run on a
benchmark's database copy (`:from_session` keeps a run's own sessions). The adherence
query, as an example:

In [ ]:
print(query("eval_adherence"))

**Vocabulary adherence** is the core promise: a reply may use at most one word outside the
learner's vocabulary (the *i+1* rule), and if a draft uses more, it gets one rewrite.
**Teaching completeness**: every new word in a reply is taught on that reply.

In [ ]:
print(metrics.format_report(metrics.report(conn)))

All 25 real replies stayed within the limit, and every reply with a new word taught it. But
the intervals are wide: 25 of 25 is consistent with a true rate as low as 87%, and 5 of 5
with 57%. These numbers will tighten as Jason uses the app; the reading and recommendation
metrics (n = 0) start once he reads in it.

Instrumentation fixed on the way (slice 4.4a): a word looked up in a conversation used to be
logged on the last tutor reply, where completeness would count it as taught by the reply.
It now gets a one-word `study` turn of its own, as in the reader.

## 2. The conversation benchmark

The real log fills slowly, so `evaluation/benchmark.json` scripts 8 topics × 5 learner
messages. 24 carry one planted mistake, labeled as the grader classifies it: a **wrong form**
of the word meant (gender, number, conjugation, tense: credited at a lower grade) or a
**wrong word** (*ser* for *estar*, *jugo* for *juego*, an English word: never credited).
16 are correct Spanish, to count false alarms. Each run uses a copy of a frozen snapshot of
the word bank, so every run faces the same vocabulary. Claude's misuse flags aren't stored
in the database, so the runner records them.

In [ ]:
runs = sorted(p for p in benchmark.RUNS_DIR.iterdir() if p.is_dir())
results = [r for run in runs for r in benchmark.read_results(run)]
grading = benchmark.score_grading(results)
print(benchmark.format_grading(grading))
for run in runs:
    s = benchmark.score(run)["summary"]
    print(f"{run.name}: {len(s['topics'])} topics, {s['cents']}¢")

In [ ]:
pd.DataFrame(
    [
        (r["topic"], r["text"], (r["error"] or {}).get("written", ""), (r["error"] or {}).get("kind", "correct"),
         ", ".join(f"{f['written']} ({'word' if f['wrong_word'] else 'form'})" for f in r["flags"]))
        for r in results
    ],
    columns=["topic", "learner message", "planted", "kind", "flagged by the grader"],
).head(12)

Every planted mistake was flagged, each as the right kind, with no false alarm on correct
messages; every reply stayed within the limit. The traps worked too: *pero* flagged only
where it meant *perro*, the correct *es muy alto* left alone beside the wrong *es enfermo*.

**The honest reading:** a perfect score also means this set is too easy to find the grader's
limits. It's a regression test for prompt changes. A harder version would need subtler
mistakes (subjunctive, *por/para*, two mistakes in a message).

## 3. The new-word detector

When a reply uses a word outside the word bank, the tutor teaches it. Is it really new?
Jason rated the 13 words taught as new (5 in his conversations, 8 in the benchmark):

In [ ]:
precision = ratings.new_word_precision(conn)
print("all:      ", precision["all"])
print("real log: ", precision["real"])
print("benchmark:", precision["benchmark"])

rank = {(r["lemma"], r["pos"]): n for n, r in enumerate(conn.execute(
    "SELECT lemma, pos FROM lexemes WHERE frequency_per_million IS NOT NULL "
    "ORDER BY frequency_per_million DESC"), 1)}
pd.DataFrame(
    [(i["content"]["lemma"], i["content"]["pos"], i["label"], rank.get((i["content"]["lemma"], i["content"]["pos"])),
      "benchmark" if i["source_ref"].startswith("benchmark:") else "real")
     for i in ratings.queue(conn, "new_word_flag")],
    columns=["word", "POS", "Jason", "frequency rank", "from"],
).sort_values("frequency rank")

62% of the words taught as new were new to Jason. None was a tagging error: the detector
reads the text correctly. The five he already knew are a **word-bank coverage** gap: four
rank between 1,821 and 2,656, past the 1,500 words the seed review offered, and *cada*
(rank 231) was offered but left blank. My own guess beforehand was wrong: I expected *qué*
(as in "¡Qué bien!"), *como* and *en casa* to be false flags; Jason marked them new.
The gap closes itself (each word taught joins the bank); a seed review of ranks 1,501-3,000
would close it faster.

## 4. Translation judges: consistent first, then compared with the learner

The lyrics skill gives a natural and a literal translation of every line. Can a model judge
the natural one? Claude Sonnet 5.5 scored 88 lines (the 10 poems within the difficulty
ceiling) on **naturalness** and **faithfulness**, 1-5, five times each. Agreement across the
repeats is measured with **Krippendorff's alpha** (`evaluation/agreement.py`, checked
against the published example: 0.743 nominal, 0.815 ordinal, 0.849 interval), which
corrects for chance and treats 4-vs-5 as a smaller disagreement than 1-vs-5.

**Rubric v2.** In a pilot the judge wobbled between 4 and 5 on "For one glance, *I'd give* a
world": is supplying a verb the Spanish implies faithful? Jason added one sentence ("English
needs many words that Spanish leaves unsaid: supplying a word the Spanish implies is
faithful"), and the wobble went:

In [ ]:
def scores(run_id, poem, criterion):
    return [
        (json.loads(r["content"])["line_no"], json.loads(r["ratings"]))
        for r in query_rows("eval_judge_consistency", run_id=run_id)
        if r["criterion"] == criterion and json.loads(r["content"])["poem"] == poem
    ]


def query_rows(name, **params):
    cursor = conn.execute(query(name), params)
    columns = [d[0] for d in cursor.description]
    return [dict(zip(columns, row)) for row in cursor]


print("v1 (pilot, run 1):", scores(1, "Rima XXIII", "faithfulness"))
print("v2 (run 3):       ", scores(3, "Rima XXIII", "faithfulness"))

In [ ]:
print(translation.format_consistency(3, translation.consistency(conn, 3)))
print()
print(translation.format_consistency(4, translation.consistency(conn, 4)))

Both criteria clear alpha 0.80, the level Krippendorff calls reliable. The attempt verdicts
(the app's own comparison, on 18 attempts written at a known quality) never changed across
repeats, and matched the intended verdict 85 times of 90. The 5 misses are one attempt:
"Do you know when it comes back?" for *¿Sabes tú adónde va?*, called "close" every time
although its own comment says the meaning is wrong. **A perfectly consistent judge can be
consistently wrong.**

**Calibration.** Consistency says the judge agrees with itself. Does it agree with the
learner? Jason rated the naturalness of 28 lines (three whole poems), without seeing the
judge's scores:

In [ ]:
print(translation.format_calibration(3, translation.calibration(conn, 3)))

In [ ]:
print(translation.format_calibration(5, translation.calibration(conn, 5)))

It doesn't. The judge is about a point stricter (3.93 against Jason's 4.86), and it marks down
lines that are fragments of a sentence, which Jason reads as natural in the poem. Rubric v3
(Jason's rule: read a line with the line before and after it, and don't judge sound) made it
**worse** (run 5): told to read whole sentences, the judge scored Bécquer's long inverted
sentences as convoluted English, an editor's standard where Jason reads as a learner. v2
was kept; v3 stays in the log as evidence.

The conclusion: the judge's naturalness scores are reliable (alpha 0.85 against itself) but
not valid for this learner, and a consistency-only evaluation would have reported them.
Poems are also the hardest case (inverted syntax, lines that split sentences) and only the
public-domain demo, so calibration is to be redone on song lyrics. Until then, translation
naturalness is reported from Jason's own ratings: **4.86 / 5 over 28 lines**.

## 5. Retrieval: RAGAS's two metrics, implemented directly

RAGAS can't be installed alongside this project's pinned models (its dependencies need
`huggingface-hub` ≥ 1.0; the pinned `transformers` needs < 1.0), so its two retrieval metrics
are implemented with a judge:

- **Context relevance** of the example sentences each conversation reply is given (the
  benchmark's 40 messages, retrieval re-run locally).
- **Faithfulness** of the talk about a text: two scripted discussions of a book's first
  chapters, each reply's claims checked against what the tutor was given. Two of the ten
  questions are traps the text doesn't answer.

In [ ]:
print("Retrieved passages (run 6):")
print(retrieval.format_summary(retrieval.summarize(conn, 6)))

The example sentences were on target (67% relevant, 99% at least partly). The talk never
invented a fact, and both traps got honest answers ("La historia no dice su nombre"). But
the three retrieved passages held the answer to only 5 of 10 questions. The chapters are
short (about 400 words), so retrieval added risk and saved nothing: **texts up to 2,000
words now go into the discussion's cached prompt whole** (`reading.WHOLE_TEXT_WORDS`).
Re-measured:

In [ ]:
print("Whole text (run 7):")
print(retrieval.format_summary(retrieval.summarize(conn, 7)))

## Summary

| Measure | Result | n | Note |
|---|---|---|---|
| Adherence (real log) | 100% | 25 replies | 95% CI 87-100% |
| Teaching completeness (real log) | 100% | 5 replies | CI 57-100% |
| Grading: planted mistakes caught, right kind | 100% | 24 | too easy to find limits |
| Grading: correct messages left alone | 100% | 16 | |
| New-word detector precision | 62% | 13 words | the misses are word-bank coverage |
| Judge consistency (naturalness / faithfulness) | alpha 0.85 / 0.82 | 88 lines × 5 | reliable |
| Judge vs Jason (naturalness) | about 1 point stricter, alpha −0.37 | 28 lines | **not valid** for the learner |
| Attempt verdicts vs intended | 94% | 18 × 5 | one consistent leniency |
| Example sentences relevant | 67% (99% at least partly) | 159 | |
| Talk faithful to the text | 97% with whole texts (48% to retrieved passages) | 33 claims | both traps passed |

The whole evaluation cost about $2.57 in model calls.

In [ ]:
conn.close()